**Цель работы:**

Осуществить предварительную обработку данных csv-файла (клиенты магазина `clients.csv`), выявить и устранить проблемы в этих данных (пропуски, дубликаты, некорректные типы данных), а также выполнить группировки и построить сводные таблицы в соответствии с вариантом 14.


# 1. Загрузка набора данных


### 1.1 Описание предметной области

**Вариант № 14**  
**Набор данных:** `clients.csv` (Данные о клиентах магазина)

Набор данных содержит 8 ключевых признаков:
1. `ID` — уникальный идентификатор клиента;
2. `Year_Birth` — год рождения клиента;
3. `Education` — уровень образования;
4. `Marital_Status` — семейное положение;
5. `Income` — годовой доход семьи;
6. `Kidhome` — количество детей в семье;
7. `Dt_Customer` — дата регистрации клиента в компании;
8. `NumDealsPurchases` — количество покупок со скидкой.


In [105]:
import pandas as pd
import numpy as np
from IPython.core import display_trap

# Загрузка датасета с разделителем ';'
df = pd.read_csv('clients.csv', sep=';')
print(f"Размер исходного датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")


Размер исходного датасета: 796 строк, 8 столбцов


### 1.2 Первые 20 строк набора данных


In [106]:
df.head(20)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Dt_Customer,NumDealsPurchases
0,5524,1957,Graduation,Single,58138.0,0.0,04.09.2012,3.0
1,2174,1954,Graduation,Single,46344.0,1.0,08.03.2014,2.0
2,4141,1965,Graduation,Together,71613.0,0.0,21.08.2013,1.0
3,6182,1984,Graduation,Together,26646.0,1.0,10.02.2014,2.0
4,5324,1981,PhD,Married,58293.0,1.0,19.01.2014,5.0
5,7446,1967,Master,Together,62513.0,0.0,09.09.2013,2.0
6,965,1971,Graduation,Divorced,55635.0,0.0,13.11.2012,4.0
7,6177,1985,PhD,Married,33454.0,1.0,08.05.2013,2.0
8,4855,1974,PhD,Together,30351.0,1.0,06.06.2013,1.0
9,5899,1950,PhD,Together,5648.0,1.0,13.03.2014,1.0


# 2. Обзор и оценка данных


### 2.1 Общая информация о данных с помощью метода info()


In [107]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 796 entries, 0 to 795
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 796 non-null    int64  
 1   Year_Birth         796 non-null    int64  
 2   Education          796 non-null    str    
 3   Marital_Status     796 non-null    str    
 4   Income             784 non-null    float64
 5   Kidhome            795 non-null    float64
 6   Dt_Customer        795 non-null    str    
 7   NumDealsPurchases  795 non-null    float64
dtypes: float64(3), int64(2), str(3)
memory usage: 49.9 KB


### 2.2 Статистическое описание числовых столбцов с помощью метода describe()


In [108]:
df.describe()


,ID,Year_Birth,Income,Kidhome,NumDealsPurchases
count,796.000000,796.000000,784.00000,795.000000,795.000000
mean,5630.133166,1968.356784,53130.07398,0.438994,2.314465
std,3273.039715,12.022132,21818.56876,0.547252,1.941650
min,0.000000,1899.000000,2447.00000,0.000000,0.000000
25%,2853.000000,1959.000000,36141.75000,0.000000,1.000000
50%,5563.000000,1969.500000,52372.50000,0.000000,2.000000
75%,8584.250000,1977.000000,69293.25000,1.000000,3.000000
max,11191.000000,1995.000000,162397.00000,2.000000,15.000000


---
**Выводы по первичному обзору данных:**
1. В датасете 796 записей и 8 столбцов.
2. Пропуски зафиксированы в столбцах `Income` (12 пропущенных значений), а также по 1 пропуску в `Kidhome`, `Dt_Customer` и `NumDealsPurchases`.
3. Столбец `Dt_Customer` представлен строковым типом `object` (формат ДД.ММ.ГГГГ), его необходимо преобразовать в `datetime64`.
4. Названия столбцов содержат заглавные буквы и стиль CamelCase, их необходимо привести к единому регистру `snake_case`.
---


### 2.3 Исследование статистических аномалий и артефактов ввода


In [109]:
# 1. Анализ неправдоподобного года рождения
outliers_age = df[df['Year_Birth'] < 1920][['ID', 'Year_Birth', 'Education', 'Marital_Status']]
print(f"Количество записей с аномальным годом рождения: {len(outliers_age)}")
print(outliers_age)

# 2. Анализ распределения доходов
print(f"\nМинимальный доход: {df['Income'].min():.2f}")
print(f"Максимальный доход: {df['Income'].max():.2f}")
print(f"Медианный доход: {df['Income'].median():.2f}")
print(f"Средний доход: {df['Income'].mean():.2f}")


Количество записей с аномальным годом рождения: 1
       ID  Year_Birth Education Marital_Status
313  1150        1899       PhD       Together

Минимальный доход: 2447.00
Максимальный доход: 162397.00
Медианный доход: 52372.50
Средний доход: 53130.07


**Интерпретация выявленных аномалий:**  
1. Год рождения 1899 (клиент ID 1150) указывает на возраст свыше 114 лет на момент сбора данных, что является явной опечаткой или артефактом ручного ввода даты.
2. Диапазон доходов варьируется от 2 447 до 162 397 у.е. со средним значением 53 130.07 и медианой 52 372.50 у.е., экстремальных выбросов, искажающих порядок величин, не зафиксировано.


### 2.4 Оценка и нормализация названий столбцов


In [110]:
print("Исходные названия столбцов:")
print(df.columns.tolist())

# Приведение названий столбцов к нижнему регистру (snake_case)
df.columns = df.columns.str.lower()
print("\nОбновленные названия столбцов:")
print(df.columns.tolist())


Исходные названия столбцов:
['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Dt_Customer', 'NumDealsPurchases']

Обновленные названия столбцов:
['id', 'year_birth', 'education', 'marital_status', 'income', 'kidhome', 'dt_customer', 'numdealspurchases']


# 3. Проверка и обработка дубликатов


### 3.1 Поиск и удаление явных дубликатов


In [111]:
explicit_duplicates = df.duplicated().sum()
print(f"Количество явных полных дубликатов: {explicit_duplicates}")
if explicit_duplicates > 0:
    print("Примеры повторяющихся строк:")
    print(df[df.duplicated(keep=False)].head(6))

# Удаление полных дубликатов с обновлением индексов
df = df.drop_duplicates().reset_index(drop=True)
print(f"Размер датасета после удаления явных дубликатов: {df.shape[0]} строк")


Количество явных полных дубликатов: 4
Примеры повторяющихся строк:
       id  year_birth   education marital_status   income  kidhome  \
791  2853        1980  Graduation         Single  51766.0      1.0   
792  2853        1980  Graduation         Single  51766.0      1.0   
793  2853        1980  Graduation         Single  51766.0      1.0   
794  2853        1980  Graduation         Single  51766.0      1.0   
795  2853        1980  Graduation         Single  51766.0      1.0   

    dt_customer  numdealspurchases  
791  11.03.2014                2.0  
792  11.03.2014                2.0  
793  11.03.2014                2.0  
794  11.03.2014                2.0  
795  11.03.2014                2.0  
Размер датасета после удаления явных дубликатов: 792 строк


### 3.2 Проверка и устранение неявных дубликатов в категориальных столбцах


In [112]:
print("Уникальные значения в столбце education:")
print(df['education'].value_counts())

print("\nУникальные значения в столбце marital_status:")
print(df['marital_status'].value_counts())


Уникальные значения в столбце education:
education
Graduation    440
PhD           190
Master        147
Basic          15
Name: count, dtype: int64

Уникальные значения в столбце marital_status:
marital_status
Married     305
Together    193
Single      174
Divorced     85
Widow        29
Alone         3
MARRIED       2
SINGL         1
Name: count, dtype: int64


**Пояснения к обработке неявных дубликатов:**  
В столбце `marital_status` выявлены следующие артефакты ввода данных:
1. `MARRIED` — написание заглавными буквами (приводится к стандартному `Married`).
2. `SINGL` — опечатка с пропущенной конечной буквой (приводится к `Single`).
3. `Alone` — смысловой синоним холостого положения (приводится к `Single`).


In [113]:
# Устранение неявных дубликатов в marital_status
df['marital_status'] = df['marital_status'].replace({
    'MARRIED': 'Married',
    'SINGL': 'Single',
    'Alone': 'Single'
})
print("Обновленное распределение по семейному положению:")
print(df['marital_status'].value_counts())


Обновленное распределение по семейному положению:
marital_status
Married     307
Together    193
Single      178
Divorced     85
Widow        29
Name: count, dtype: int64


# 4. Проверка и обработка пропусков


In [114]:
missing_values = df.isna().sum()
print("Количество пропусков по столбцам:")
print(missing_values[missing_values > 0])


Количество пропусков по столбцам:
income               12
kidhome               1
dt_customer           1
numdealspurchases     1
dtype: int64


**Пояснения к обработке пропусков:**  
1. В строке с ID 1994 (индекс 10) пропущены значения сразу в 4 ключевых столбцах: `income`, `kidhome`, `dt_customer` и `numdealspurchases` (более 50% целевой информации объекта). Восстановление даты регистрации и покупок внесло бы искажение, поэтому данная единичная дефектная запись удалена в соответствии с методикой устранения неинформативных строк.
2. В столбце `income` после удаления неинформативной строки осталось 11 пропусков. Они заполнены медианным значением дохода ($52,372.50), которое устойчиво к асимметрии распределения.


In [115]:
# Удаление неинформативной строки с пропущенной датой и целевыми атрибутами
df = df.dropna(subset=['dt_customer']).reset_index(drop=True)

# Заполнение оставшихся пропусков в income медианным значением
median_income = df['income'].median()
df['income'] = df['income'].fillna(median_income)

print("Проверка пропусков после обработки:")
print(f"Всего пропусков в датасете: {df.isna().sum().sum()}")


Проверка пропусков после обработки:
Всего пропусков в датасете: 0


# 5. Проверка и преобразование типов данных


In [116]:
# Преобразование даты регистрации к формату datetime
df['dt_customer'] = pd.to_datetime(df['dt_customer'], format='%d.%m.%Y')

# Приведение числовых признаков к целочисленному типу
df['income'] = df['income'].round().astype('Int64')
df['kidhome'] = df['kidhome'].astype(int)
df['numdealspurchases'] = df['numdealspurchases'].astype(int)

print("Типы данных после преобразования:")
print(df.dtypes)


Типы данных после преобразования:
id                            int64
year_birth                    int64
education                       str
marital_status                  str
income                        Int64
kidhome                       int64
dt_customer          datetime64[us]
numdealspurchases             int64
dtype: object


# 6. Выполнение заданий Варианта 14


### Задание 1
**Формулировка задания:**  
Группировка — тип образования (`education`) по каждому семейному статусу (`marital_status`).


In [117]:
task1_result = df.groupby(['education', 'marital_status'])['id'].count()
task1_result


education   marital_status
Basic       Divorced            1
            Married             9
            Single              1
            Together            4
Graduation  Divorced           49
            Married           170
            Single            103
            Together           96
            Widow              21
Master      Divorced           12
            Married            57
            Single             32
            Together           42
            Widow               4
PhD         Divorced           23
            Married            70
            Single             42
            Together           51
            Widow               4
Name: id, dtype: int64

**Интерпретация результатов Задания 1:**  
Наиболее многочисленной категорией клиентов во всех образовательных стратах выступают лица, состоящие в браке (`Married`) и проживающие совместно (`Together`). Доминирующей группой является высшее образование (`Graduation` — 170 состоящих в браке и 96 проживающих совместно). Категории клиентов со степенями `PhD` и `Master` демонстрируют схожую структуру распределения.


### Задание 2
**Формулировка задания:**  
Группировка — семейный статус (`marital_status`) по количеству детей (`kidhome`). Создать датафрейм. Переименовать столбец с количеством в `"count"`. Отсортировать по убыванию столбца `"count"`.


In [118]:
task2_df = df.groupby(['marital_status', 'kidhome'])['id'].count().reset_index(name='count')
task2_df = task2_df.sort_values(by='count', ascending=False).reset_index(drop=True)
task2_df


,marital_status,kidhome,count
0,Married,0,169
1,Married,1,125
2,Together,0,124
3,Single,0,98
4,Single,1,76
5,Together,1,66
6,Divorced,0,52
7,Divorced,1,31
8,Widow,0,24
9,Married,2,12


**Интерпретация результатов Задания 2:**  
Лидирующими сегментами аудитории являются состоящие в браке семьи без детей (169 клиентов) и с одним ребенком (125 клиентов), а также пары в фактическом браке без детей (124 клиента). Семьи с двумя детьми представлены в существенно меньшем объеме (12 семей `Married` и по 2–4 в остальных категориях).


### Задание 3
**Формулировка задания:**  
Сводная таблица (`pivot_table`) — средний доход семьи по семейному положению (`marital_status`). Отсортировать по убыванию. Округлить до двух знаков.


In [119]:
# Основная реализация с помощью pivot_table согласно заданию
task3_pivot = df.pivot_table(index='marital_status', values='income', aggfunc='mean').round(2)
task3_pivot = task3_pivot.sort_values(by='income', ascending=False)
task3_pivot


,income
marital_status,
Widow,55452.45
Divorced,54568.16
Together,54493.05
Married,52894.63
Single,50990.66


**Сравнение методов `pivot_table` и `groupby`:**  
Для одномерной агрегации по одной категориальной переменной идиоматичным и производительным решением в Pandas является метод `groupby`:  
`df.groupby('marital_status')['income'].mean().round(2).sort_values(ascending=False).to_frame(name='income')`.  
Метод `pivot_table` использован для строгого следования условию задания на построение сводной таблицы.

**Интерпретация результатов Задания 3:**  
Наибольшим средним доходом обладают клиенты категории `Widow` (55 452.45 у.е.), `Divorced` (54 568.16 у.е.) и `Together` (54 493.05 у.е.). Категория `Single` имеет наименьший средний показатель дохода (50 990.66 у.е.).


### Задание 4
**Формулировка задания:**  
Сводная таблица (`pivot_table`) — среднее количество покупок (`numdealspurchases`) по уровню образования (`education`) — строки и году рождения (`year_birth`) — столбцы. Отсортировать по возрастанию `education`. Округлить до двух знаков.


In [120]:
    # Построение сводной таблицы с устранением структурных пустот (fill_value=0)
task4_pivot = df.pivot_table(
    index='education', 
    columns='year_birth', 
    values='numdealspurchases', 
    aggfunc='mean',
    fill_value=0
).round(2)
task4_pivot = task4_pivot.sort_index(ascending=True)
task4_pivot


year_birth,1899,1941,1943,1944,1945,1946,1947,1948,1949,1950,...,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995
education,,,,,,,,,,,,,,,,,,,,,
Basic,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,2.00,2.00,0.00,1.00,1.0,0.0,0.0,0.0,0.0,0.0
Graduation,0.0,0.0,0.0,1.0,0.0,1.67,1.0,1.00,1.0,2.00,...,1.12,1.67,1.92,1.78,1.0,1.0,1.0,1.0,1.0,1.0
Master,0.0,0.0,1.0,1.0,1.0,1.00,1.0,1.00,2.2,5.00,...,2.50,1.00,1.00,2.00,0.0,0.0,1.0,0.0,0.0,0.0
PhD,1.0,0.0,1.5,1.0,1.5,2.00,1.0,2.43,2.5,1.67,...,1.50,0.00,1.00,1.00,0.0,1.0,0.0,0.0,0.0,0.0


**Обоснование обработки матрицы и интерпретация результатов Задания 4:**  
1. *Устранение структурных пустот:* В разреженной двумерной матрице исходно возникают значения `NaN` из-за отсутствия в выборке наблюдений для некоторых комбинаций года рождения и образования. Использование параметра `fill_value=0` корректно заменяет пустые ячейки нулями, отображая отсутствие совершенных покупок.  
2. *Интерпретация:* Клиенты с учеными степенями (`PhD`, `Master`) и высшим образованием (`Graduation`) более активно совершают покупки со скидкой (до 2.4–4.0 покупок в среднем на клиента по когортам), в то время как клиенты со статусом `Basic` совершают минимум акционных покупок.


# 7. Заключение (Выводы)

В ходе выполнения лабораторной работы выполнен предварительный анализ данных набора клиентов магазина `clients.csv` (796 наблюдений, 8 признаков) в соответствии с вариантом 14:

1. **Качество данных и предварительная обработка:**  
   В ходе первичного анализа выявлено 4 полных дубликата строк, которые были удалены. Обнаружена одна неинформативная строка (ID 1994, потеря >50% ключевых атрибутов), исключенная из анализа, и 11 пропусков в столбце `income`, заполненных медианным значением (52 372.50 у.е.). В категориальном столбце `marital_status` успешно устранены неявные дубликаты (`MARRIED`, `SINGL`, `Alone` сведены к стандартным категориям). Дата регистрации приведена к типу `datetime64`, числовые атрибуты приведены к целочисленным типам.

2. **Результаты группировок и демографический портрет аудитории:**  
   Группировки показали, что доминирующую долю покупателей составляют лица с высшим образованием (`Graduation` — 170 состоящих в браке, 103 холостых, 96 проживающих совместно). Основным клиентским ядром являются семьи без детей (169 семей в браке и 124 в союзе) и семьи с одним ребенком (125 в браке). Анализ среднего дохода показал лидерство групп `Widow` (55 452.45 у.е.) и `Divorced` (54 568.16 у.е.), тогда как холостые клиенты (`Single`) обладают наименьшим средним доходом (50 990.66 у.е.).

3. **Сводные таблицы и анализ потребительской активности:**  
   Построена сводная таблица зависимости числа покупок со скидкой (`numdealspurchases`) от уровня образования и года рождения. Возникавшие структурные пустоты матрицы кросс-табуляции устранены с помощью параметра `fill_value=0`. Анализ подтвердил, что покупатели со степенями `PhD` и `Master` проявляют наибольшую активность в акционных покупках.



# Дополнительные задания 12 и 21

В следующих ячейках выполняются категоризация дохода и возраста, фильтрация клиентов по заданным условиям и построение сводных таблиц. Все расчеты выполняются на очищенном датафрейме `df`.

## Задание 12. Категория дохода и сводные таблицы

Для границ категорий дохода используются 33-й и 67-й процентили. Такой подход выбран потому, что он делит наблюдения примерно на три сопоставимые по численности группы и не зависит от произвольных денежных порогов. Доход ниже первого порога считается низким, между порогами — средним, выше второго — высоким.

In [121]:
# Категоризация дохода по терцилям
income_q33 = df['income'].quantile(1 / 3)
income_q67 = df['income'].quantile(2 / 3)

def income_category(value):
    if value < income_q33:
        return 'Низкий'
    if value <= income_q67:
        return 'Средний'
    return 'Высокий'

df['income_category'] = df['income'].apply(income_category)
print(f'Порог низкого дохода: менее {income_q33:.2f}')
print(f'Порог высокого дохода: более {income_q67:.2f}')
print('Распределение по категориям:')
display(df['income_category'].value_counts().rename_axis('income_category').to_frame('count'))

Порог низкого дохода: менее 42243.00
Порог высокого дохода: более 64029.00
Распределение по категориям:


,count
income_category,
Высокий,264
Низкий,264
Средний,263


In [122]:
# Средний и медианный доход по категории дохода и образованию
task12_pivot = df.pivot_table(
    index='income_category',
    columns='education',
    values='income',
    aggfunc=['mean', 'median']
).round(2)
task12_pivot

mean                                  median             \
education          Basic Graduation    Master       PhD    Basic Graduation   
income_category                                                               
Высокий             <NA>   76617.13  76958.41   77659.0     <NA>    75049.5   
Низкий           21235.0   29197.85  29850.07  32690.65  24279.0    30096.0   
Средний             <NA>   53030.66  52684.97  53248.39     <NA>    52614.0   

                                   
education         Master      PhD  
income_category                    
Высокий          76995.0  74165.0  
Низкий           31160.0  34320.0  
Средний          52614.0  52614.0

## Задание 21. Возраст, расширенная категоризация и фильтрация

Возраст рассчитывается как разница между 2014 годом и `year_birth`: 2014 выбран как единый контрольный год, соответствующий периоду набора данных. Границы категорий: молодой — до 40 лет, средний — от 40 до 59 лет, старый — 60 лет и старше. Эти интервалы позволяют отделить условно активную молодую группу, основную группу среднего возраста и старшую группу.

Для фильтрации дополнительно будут выбраны два семейных статуса и два типа образования с наименьшим средним доходом. Это реализует условие «топ 2» в смысле двух нижних позиций рейтинга по среднему доходу.

In [123]:
# Расчет возраста и его категоризация
reference_year = 2014
df['age'] = reference_year - df['year_birth']

def age_category(age):
    if age < 40:
        return 'Молодой'
    if age < 60:
        return 'Средний'
    return 'Старый'

df['age_category'] = df['age'].apply(age_category)

# Повторно фиксируем категории дохода как часть задания 21
df['income_category'] = df['income'].apply(income_category)

print('Распределение по категориям возраста:')
display(df['age_category'].value_counts().rename_axis('age_category').to_frame('count'))

status_income = df.groupby('marital_status')['income'].mean().sort_values()
education_income = df.groupby('education')['income'].mean().sort_values()
lowest_statuses = status_income.head(2).index.tolist()
lowest_educations = education_income.head(2).index.tolist()

print('Два семейных статуса с минимальным средним доходом:', lowest_statuses)
print('Два типа образования с минимальным средним доходом:', lowest_educations)

filtered_clients = df[
    (df['income_category'] == 'Низкий')
    & (df['age_category'] == 'Средний')
    & (df['marital_status'].isin(lowest_statuses))
    & (df['education'].isin(lowest_educations))
].copy()
print(f'Количество записей после комплексной фильтрации: {len(filtered_clients)}')
filtered_clients.head()

Распределение по категориям возраста:


,count
age_category,
Средний,409
Молодой,262
Старый,120


Два семейных статуса с минимальным средним доходом: ['Single', 'Married']
Два типа образования с минимальным средним доходом: ['Basic', 'Graduation']
Количество записей после комплексной фильтрации: 42


,id,year_birth,education,marital_status,income,kidhome,dt_customer,numdealspurchases,income_category,age,age_category
23,7892,1969,Graduation,Single,18589,0,2013-01-02,2,Низкий,45,Средний
61,8082,1971,Graduation,Married,25721,1,2013-05-21,1,Низкий,43,Средний
74,2261,1969,Graduation,Married,26304,1,2013-06-23,1,Низкий,45,Средний
77,5268,1960,Graduation,Married,29440,1,2013-08-11,2,Низкий,54,Средний
167,2563,1961,Basic,Married,28249,0,2014-06-15,1,Низкий,53,Средний


In [124]:
# Сводные таблицы на отфильтрованных данных:
# средний и медианный доход по образованию и семейному статусу
task21_pivot = filtered_clients.pivot_table(
    index=['education', 'marital_status'],
    values='income',
    aggfunc=['mean', 'median']
).round(2)
task21_pivot

mean   median
                             income   income
education  marital_status                   
Basic      Married          23613.5  23613.5
Graduation Married         27419.04  26751.0
           Single           24528.0  22804.0